In [ ]:
# === 安裝必要套件 ===
# pip install pydub gradio_client
# 注意自己要放 S01_P01.wav 音檔在當前目錄

In [2]:
from pydub import AudioSegment
from gradio_client import Client, file
import re
import os

transcript_path = "transcript_with_speakers.txt"
audio_path = "360.mp3"  # 可以是 .wav 或 .mp3
output_dir = "first_utterances/"
os.makedirs(output_dir, exist_ok=True)

# === 讀取 transcript ===
with open(transcript_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# === 擷取第一句發言（條件：長度 > 1.5 秒）===
speaker_first_utterance = {}

for line in lines:
    match = re.match(r"\[(\d+\.\d+)s - (\d+\.\d+)s\] (SPEAKER_\d+): (.+)", line)
    if match:
        start_time = float(match.group(1))
        end_time = float(match.group(2))
        speaker = match.group(3)
        text = match.group(4).strip().lower()
        duration = end_time - start_time

        if speaker not in speaker_first_utterance:
            if duration <= 1.5:
                continue
            speaker_first_utterance[speaker] = (start_time, end_time)

# === 根據副檔名自動載入音檔 ===
ext = os.path.splitext(audio_path)[-1].lower()

if ext == ".wav":
    audio = AudioSegment.from_wav(audio_path)
elif ext == ".mp3":
    audio = AudioSegment.from_mp3(audio_path)
else:
    raise ValueError(f"不支援的音訊格式：{ext}")

# === 輸出每位講者的第一句發言 ===
for speaker, (start, end) in speaker_first_utterance.items():
    start_ms = int(start * 1000)
    end_ms = int(end * 1000)
    segment = audio[start_ms:end_ms]
    output_file = os.path.join(output_dir, f"{speaker}_first.wav")
    segment.export(output_file, format="wav")
    print(f"✅ Saved: {output_file}")


✅ Saved: first_utterances/SPEAKER_11_first.wav
✅ Saved: first_utterances/SPEAKER_03_first.wav
✅ Saved: first_utterances/SPEAKER_08_first.wav
✅ Saved: first_utterances/SPEAKER_00_first.wav
✅ Saved: first_utterances/SPEAKER_07_first.wav
✅ Saved: first_utterances/SPEAKER_06_first.wav
✅ Saved: first_utterances/SPEAKER_16_first.wav
✅ Saved: first_utterances/SPEAKER_04_first.wav
✅ Saved: first_utterances/SPEAKER_05_first.wav
✅ Saved: first_utterances/SPEAKER_13_first.wav
✅ Saved: first_utterances/SPEAKER_17_first.wav
✅ Saved: first_utterances/SPEAKER_09_first.wav
✅ Saved: first_utterances/SPEAKER_01_first.wav
✅ Saved: first_utterances/SPEAKER_02_first.wav
✅ Saved: first_utterances/SPEAKER_10_first.wav
✅ Saved: first_utterances/SPEAKER_14_first.wav
✅ Saved: first_utterances/SPEAKER_15_first.wav


In [3]:
# 建立 Gradio Space client
client = Client("marsyao/voice-gender-classifier")

folder_path = "first_utterances"
wav_files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

# 執行性別分類
for wav_file in sorted(wav_files):
    file_path = os.path.join(folder_path, wav_file)

    # 取 speaker 名稱 (如 SPEAKER_00)
    speaker_name = "_".join(wav_file.split("_")[:2])

    try:
        result = client.predict(
            filepath=file(file_path),
            api_name="/predict"
        )
        label = result["label"].lower().replace("human ", "")
        print(f"{speaker_name} → {label}")
    except Exception as e:
        print(f"{speaker_name} → 發生錯誤：{e}")


Loaded as API: https://marsyao-voice-gender-classifier.hf.space ✔


c:\Users\c1091\anaconda3\envs\audia\lib\site-packages\gradio_client\utils.py:1228: UserWarning: file() is deprecated and will be removed in a future version. Use handle_file() instead.
  warnings.warn(


SPEAKER_00 → male
SPEAKER_01 → male
SPEAKER_02 → female
SPEAKER_03 → female
SPEAKER_04 → female
SPEAKER_05 → female
SPEAKER_06 → female
SPEAKER_07 → female
SPEAKER_08 → female
SPEAKER_09 → female
SPEAKER_10 → male
SPEAKER_11 → male
SPEAKER_13 → female
SPEAKER_14 → male
SPEAKER_15 → male
SPEAKER_16 → male
SPEAKER_17 → female
